# Computationally Efficient Transformer dengan Cost-Sensitive Attention + Adaptive Focal Loss
### untuk Deteksi Serangan Kelas Minoritas pada IDS (NSL-KDD / CICIDS2017)

Notebook ini siap dijalankan di **Google Colab** (Runtime → Change runtime type → GPU disarankan, tapi CPU juga bisa).

**Dataset dapat dipilih** melalui variabel `DATASET_CHOICE`:
- `"NSLKDD"` — diunduh otomatis dari mirror GitHub publik, ringan dan cepat untuk eksperimen awal.
- `"CICIDS2017"` — diunduh via Kaggle API (butuh `kaggle.json`), lebih besar dan lebih realistis, dengan kelas minoritas ekstrem (Heartbleed, Infiltration).

Alur notebook:
1. Setup & pilih + unduh dataset
2. Praproses data (tanpa augmentasi sintetis — SMOTE/GAN sengaja tidak dipakai pada model utama)
3. Model yang diusulkan: Feature Tokenizer → Lightweight Transformer Encoder → **Class-Specific Attention Pooling** (implementasi cost-sensitive attention) → classifier
4. **Adaptive Focal Loss** (berbasis effective number of samples, Cui et al. 2019)
5. Baseline pembanding: Transformer standar (mean-pooling + cross-entropy biasa)
6. Training kedua model
7. Evaluasi: precision/recall/F1 per kelas, macro-F1, G-mean (fokus otomatis ke kelas paling minoritas)
8. **Profil efisiensi komputasi**: parameter, FLOPs, memory footprint, latensi single-thread, dan dynamic quantization
9. Studi ablasi
10. **Uji signifikansi statistik multi-seed** (mean ± std + Wilcoxon signed-rank test)
11. **Perbandingan dengan SMOTE** sebagai baseline penyeimbangan data konvensional
12. **Analisis attention (XAI)** — fitur mana yang ditarik oleh query tiap kelas

> **Catatan framing:** klaim efisiensi pada notebook ini adalah *computational efficiency* yang diukur pada simulasi CPU terbatas (single-thread), **bukan** hasil pengujian pada perangkat edge fisik. Jangan mengklaim "edge deployment" di paper tanpa pengujian hardware yang sebenarnya.

> Catatan NSL-KDD: dataset diunduh dari mirror publik GitHub. Jika link putus, unggah manual `KDDTrain+.txt` dan `KDDTest+.txt` ke Colab lalu lewati sel unduhan.
> Catatan CICIDS2017: dataset diunduh via Kaggle API — perlu `kaggle.json` (Kaggle → Account → Create New API Token). Gunakan `SAMPLE_FRAC` jika ingin mempercepat eksperimen.

> ### ⚠️ Jika environment Colab sudah terlanjur rusak
>
> Gejala: muncul daftar panjang `dependency conflicts`, lalu `AttributeError`
> saat `from sklearn.preprocessing import ...`.
>
> Penyebab: menjalankan `pip install ... --upgrade` di Colab menaikkan versi
> `pandas`, `numpy`, dan `torch` melampaui versi yang didukung Colab.
>
> **Cara memulihkan:** menu **Runtime → Disconnect and delete runtime**,
> lalu buka ulang notebook dan jalankan dari sel pertama. Merestart runtime saja
> tidak cukup, karena paket yang terlanjur ter-upgrade masih tersimpan di disk.
>
> Notebook ini sudah tidak lagi meng-upgrade paket apa pun.


## 1. Setup

In [ ]:
# PENTING: JANGAN gunakan --upgrade di Colab.
# torch, scikit-learn, pandas, numpy, dan matplotlib sudah terpasang di Colab.
# Meng-upgrade paket-paket tersebut akan merusak environment bawaan Colab
# (pandas/numpy/torch versi baru tidak cocok dengan paket CUDA milik Colab).
# Sel ini hanya melakukan import, tanpa instalasi apa pun.

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
import time, math, random
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
import sklearn; print("scikit-learn:", sklearn.__version__)


## 2. Pilih & Muat Dataset

In [ ]:
# ================================================================
# UBAH DI SINI untuk berpindah dataset: "NSLKDD" atau "CICIDS2017"
#
# PENTING: sel-sel CICIDS2017 dibungkus `if DATASET_CHOICE == "CICIDS2017"`,
# sehingga bila nilai di bawah masih "NSLKDD", sel-sel tersebut dilewati
# TANPA pesan error apa pun. Periksa output sel praproses: jika kelas yang
# muncul lima kategori (DoS/Normal/Probe/R2L/U2R), yang berjalan NSL-KDD;
# CICIDS2017 menghasilkan sembilan kategori.
#
# Untuk menjalankan CICIDS2017: ganti nilai di bawah, lalu Runtime ->
# Restart session and run all. Siapkan kaggle.json.
# ================================================================
DATASET_CHOICE = "CICIDS2017"

TAG = DATASET_CHOICE.lower()   # akhiran nama berkas keluaran
print("Dataset dipilih:", DATASET_CHOICE)
if DATASET_CHOICE == "CICIDS2017":
    print("Subsampling: CAP_PER_CLASS = 50.000 baris per kelas (kelas langka utuh).")
    print("Siapkan kaggle.json — sel unduhan akan meminta upload.")

### 2A. NSL-KDD (aktif jika `DATASET_CHOICE == "NSLKDD"`)

In [ ]:
import urllib.request, os

if DATASET_CHOICE == "NSLKDD":
    TRAIN_URL = "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain%2B.txt"
    TEST_URL  = "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest%2B.txt"

    os.makedirs("data", exist_ok=True)
    train_path, test_path = "data/KDDTrain+.txt", "data/KDDTest+.txt"

    try:
        if not os.path.exists(train_path):
            urllib.request.urlretrieve(TRAIN_URL, train_path)
        if not os.path.exists(test_path):
            urllib.request.urlretrieve(TEST_URL, test_path)
        print("Dataset NSL-KDD berhasil diunduh.")
    except Exception as e:
        print("Gagal mengunduh otomatis:", e)
        print("Silakan unggah manual KDDTrain+.txt dan KDDTest+.txt ke folder 'data/' lalu jalankan ulang sel ini.")

In [ ]:
if DATASET_CHOICE == "NSLKDD":
    # 41 fitur + label + difficulty level (format resmi NSL-KDD)
    columns = [
        "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
        "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
        "root_shell","su_attempted","num_root","num_file_creations","num_shells",
        "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
        "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
        "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
        "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
        "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
        "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
        "label","difficulty"
    ]

    df_train = pd.read_csv(train_path, names=columns)
    df_test  = pd.read_csv(test_path, names=columns)
    print("Train:", df_train.shape, "| Test:", df_test.shape)

    attack_map = {
        "normal": "Normal",
        # DoS
        "back": "DoS", "land": "DoS", "neptune": "DoS", "pod": "DoS", "smurf": "DoS",
        "teardrop": "DoS", "mailbomb": "DoS", "processtable": "DoS", "udpstorm": "DoS",
        "apache2": "DoS", "worm": "DoS",
        # Probe
        "ipsweep": "Probe", "nmap": "Probe", "portsweep": "Probe", "satan": "Probe",
        "mscan": "Probe", "saint": "Probe",
        # R2L
        "ftp_write": "R2L", "guess_passwd": "R2L", "imap": "R2L", "multihop": "R2L",
        "phf": "R2L", "spy": "R2L", "warezclient": "R2L", "warezmaster": "R2L",
        "sendmail": "R2L", "named": "R2L", "snmpgetattack": "R2L", "snmpguess": "R2L",
        "xlock": "R2L", "xsnoop": "R2L", "httptunnel": "R2L",
        # U2R
        "buffer_overflow": "U2R", "loadmodule": "U2R", "perl": "U2R", "rootkit": "U2R",
        "ps": "U2R", "sqlattack": "U2R", "xterm": "U2R",
    }

    def map_label_nslkdd(lbl):
        lbl = lbl.strip().lower()
        return attack_map.get(lbl, "Unknown")

    for d in (df_train, df_test):
        d["category"] = d["label"].apply(map_label_nslkdd)
        d.drop(columns=["label", "difficulty"], inplace=True)

    df_train = df_train[df_train["category"] != "Unknown"].reset_index(drop=True)
    df_test  = df_test[df_test["category"] != "Unknown"].reset_index(drop=True)

    categorical_cols = ["protocol_type", "service", "flag"]
    numeric_cols = [c for c in df_train.columns if c not in categorical_cols + ["category"]]

    print(df_train["category"].value_counts())
    print("\nDistribusi kelas (persen) - TRAIN:")
    print((df_train["category"].value_counts(normalize=True) * 100).round(2))

### 2B. CICIDS2017 (aktif jika `DATASET_CHOICE == "CICIDS2017"`)

Diunduh via Kaggle API. Langkah:
1. Buat token di [kaggle.com/settings](https://www.kaggle.com/settings) → *Create New API Token* → akan mengunduh `kaggle.json`.
2. Jalankan sel di bawah, unggah `kaggle.json` saat diminta (hanya perlu sekali per sesi Colab).
3. Dataset berukuran cukup besar (~500 MB terkompresi, jutaan baris) — atur `SAMPLE_FRAC` di sel berikutnya bila ingin mempercepat eksperimen.

In [ ]:
if DATASET_CHOICE == "CICIDS2017":
    os.makedirs("/root/.kaggle", exist_ok=True)
    kaggle_json_path = "/root/.kaggle/kaggle.json"
    if not os.path.exists(kaggle_json_path):
        try:
            from google.colab import files
            print("Silakan upload file kaggle.json (Kaggle -> Account -> Create New API Token).")
            uploaded = files.upload()
            import shutil
            for fn in uploaded:
                if fn == "kaggle.json":
                    shutil.move(fn, kaggle_json_path)
        except ImportError:
            print("Bukan lingkungan Colab — pastikan kaggle.json sudah ada di ~/.kaggle/kaggle.json secara manual.")
        if os.path.exists(kaggle_json_path):
            os.chmod(kaggle_json_path, 0o600)

    import subprocess as sp

    os.system("pip -q install kaggle")
    os.makedirs("data/cicids2017", exist_ok=True)

    # Slug lama `cicdataset/cicids2017` mengembalikan 403 Forbidden (dataset tidak lagi
    # dapat diakses publik), bukan karena token salah. Mirror di bawah memuat folder
    # MachineLearningCVE dengan delapan CSV harian berukuran identik byte-per-byte
    # dengan rilis resmi CIC, sehingga datanya sama persis.
    SLUGS = [
        "mdalamintalukder/cicids2017",   # MachineLearningCVE/ (8 CSV asli)
        "kk0105/cicids2017",             # cadangan: struktur sama, folder lebih dalam
    ]
    ok = False
    for slug in SLUGS:
        r = sp.run(["kaggle", "datasets", "download", "-d", slug,
                    "-p", "data/cicids2017", "--unzip"],
                   capture_output=True, text=True)
        if r.returncode == 0:
            print(f"CICIDS2017 berhasil diunduh & diekstrak dari '{slug}'.")
            ok = True
            break
        print(f"Gagal dari '{slug}' (rc={r.returncode}): "
              f"{(r.stdout + r.stderr).strip()[:300]}")

    if not ok:
        print("\nSeluruh mirror gagal. Periksa:")
        print("  1. kaggle.json valid (403 = dataset ditolak, 401 = token salah/kedaluwarsa)")
        print("  2. Atau unggah CSV CICIDS2017 manual ke folder 'data/cicids2017/'.")

In [ ]:
if DATASET_CHOICE == "CICIDS2017":
    import glob
    from sklearn.model_selection import train_test_split

    csv_files = glob.glob("data/cicids2017/**/*.csv", recursive=True)
    print(f"Ditemukan {len(csv_files)} file CSV CICIDS2017.")
    assert len(csv_files) > 0, "Tidak ada CSV ditemukan — pastikan unduhan/upload berhasil sebelum lanjut."

    dfs = []
    for fp in csv_files:
        d = pd.read_csv(fp, low_memory=False)
        d.columns = d.columns.str.strip()  # header resmi CICIDS2017 punya spasi di depan nama kolom
        dfs.append(d)
    df_cic = pd.concat(dfs, ignore_index=True)
    print("Total baris gabungan (seluruh hari):", df_cic.shape)

    df_cic = df_cic.replace([np.inf, -np.inf], np.nan).dropna().drop_duplicates()

    def map_label_cicids(lbl):
        lbl = str(lbl).strip()
        if lbl == "BENIGN":
            return "Normal"
        if lbl.startswith("DoS"):
            return "DoS"
        if lbl == "DDoS":
            return "DDoS"
        if lbl == "PortScan":
            return "PortScan"
        if "Patator" in lbl:
            return "BruteForce"
        if lbl.startswith("Web Attack"):
            return "WebAttack"
        if lbl == "Bot":
            return "Bot"
        if lbl == "Infiltration":
            return "Infiltration"
        if lbl == "Heartbleed":
            return "Heartbleed"
        return lbl

    df_cic["category"] = df_cic["Label"].apply(map_label_cicids)
    df_cic = df_cic.drop(columns=["Label"])
    print(df_cic["category"].value_counts())

    # Buang kelas dengan sampel terlalu sedikit untuk stratified split (dicatat, bukan dihapus diam-diam)
    counts = df_cic["category"].value_counts()
    valid_classes = counts[counts >= 10].index
    dropped = counts[counts < 10]
    if len(dropped) > 0:
        print("\nKelas berikut dibuang karena sampel < 10 (tidak cukup untuk split stratified):")
        print(dropped)
    df_cic = df_cic[df_cic["category"].isin(valid_classes)].reset_index(drop=True)

    # ---- Subsampling: BATASI KELAS MAYORITAS, PERTAHANKAN KELAS LANGKA UTUH ----
    #
    # Sampling proporsional (frac) KELIRU untuk dataset ini. CICIDS2017 memuat
    # Heartbleed (11 baris) dan Infiltration (36 baris); pada frac=0.1 Heartbleed
    # menyisakan 1 baris sehingga stratified split gagal — dan kelas paling langka,
    # yang justru menjadi objek penelitian ini, lenyap.
    #
    # Karena itu tiap kelas dipotong pada CAP_PER_CLASS baris, sedangkan kelas yang
    # jumlahnya di bawah cap dibiarkan utuh. Konsekuensinya rasio imbalance menjadi
    # lebih ringan daripada data aslinya; ini WAJIB disebutkan di bagian Metodologi,
    # bukan didiamkan.
    CAP_PER_CLASS = 50_000      # None untuk memakai seluruh data (butuh waktu jauh lebih lama)
    if CAP_PER_CLASS is not None:
        before = len(df_cic)
        df_cic = (df_cic.groupby("category", group_keys=False)
                  .apply(lambda g: g.sample(min(len(g), CAP_PER_CLASS), random_state=SEED))
                  .reset_index(drop=True))
        print(f"\nSubsampling cap {CAP_PER_CLASS:,}/kelas: {before:,} -> {len(df_cic):,} baris")
        print(df_cic["category"].value_counts())
        imb = df_cic["category"].value_counts()
        print(f"Rasio imbalance setelah cap: {imb.max() / imb.min():.0f}:1 "
              f"(kelas terkecil {imb.idxmin()} = {imb.min()} baris)")

    df_train, df_test = train_test_split(
        df_cic, test_size=0.25, stratify=df_cic["category"], random_state=SEED
    )
    df_train = df_train.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)

    categorical_cols = []  # seluruh fitur flow CICIDS2017 bersifat numerik
    numeric_cols = [c for c in df_cic.columns if c != "category"]

    print("Train:", df_train.shape, "| Test:", df_test.shape, "| n_numeric:", len(numeric_cols))

## 3. Praproses Bersama (berlaku untuk dataset manapun yang dipilih)

Label serangan pada NSL-KDD dipetakan ke 5 kategori standar (**Normal, DoS, Probe, R2L, U2R**); pada CICIDS2017 ke 9 kategori (**Normal, DoS, DDoS, PortScan, BruteForce, WebAttack, Bot, Infiltration, Heartbleed**). Kelas paling minoritas pada kedua dataset (mis. U2R pada NSL-KDD; Heartbleed/Infiltration pada CICIDS2017) menjadi fokus utama evaluasi.
**Tidak ada SMOTE/GAN** yang diterapkan di sini; ketidakseimbangan kelas ditangani oleh arsitektur model dan loss function.

In [ ]:
# Encode label kelas
class_encoder = LabelEncoder()
y_train = class_encoder.fit_transform(df_train["category"])
y_test  = class_encoder.transform(df_test["category"])
class_names = list(class_encoder.classes_)
n_classes = len(class_names)
print("Kelas:", class_names)

# Encode fitur kategorikal (index untuk embedding), tangani kategori tak terlihat di test
cat_encoders, cat_cardinalities = {}, []
X_train_cat, X_test_cat = [], []
for col in categorical_cols:
    le = LabelEncoder()
    le.fit(pd.concat([df_train[col], df_test[col]], axis=0))
    cat_encoders[col] = le
    X_train_cat.append(le.transform(df_train[col]))
    X_test_cat.append(le.transform(df_test[col]))
    cat_cardinalities.append(len(le.classes_))

if len(categorical_cols) > 0:
    X_train_cat = np.stack(X_train_cat, axis=1)
    X_test_cat  = np.stack(X_test_cat, axis=1)
else:  # dataset tanpa fitur kategorikal (mis. CICIDS2017)
    X_train_cat = np.zeros((len(df_train), 0), dtype=np.int64)
    X_test_cat  = np.zeros((len(df_test), 0), dtype=np.int64)

# Normalisasi fitur numerik
scaler = StandardScaler()
X_train_num = scaler.fit_transform(df_train[numeric_cols].astype(float))
X_test_num  = scaler.transform(df_test[numeric_cols].astype(float))

print("Numerik:", X_train_num.shape, "| Kategorikal:", X_train_cat.shape, "| Kardinalitas:", cat_cardinalities)

In [ ]:
class IDSDataset(torch.utils.data.Dataset):
    def __init__(self, X_num, X_cat, y):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_num[idx], self.X_cat[idx], self.y[idx]

train_ds = IDSDataset(X_train_num, X_train_cat, y_train)
test_ds  = IDSDataset(X_test_num, X_test_cat, y_test)

BATCH_SIZE = 256
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

n_numeric = X_train_num.shape[1]
n_categorical = X_train_cat.shape[1]
print(f"n_numeric={n_numeric}, n_categorical={n_categorical}, n_classes={n_classes}")

## 4. Arsitektur Model yang Diusulkan

**Feature Tokenizer** — tiap fitur numerik diproyeksikan (linear) menjadi token berdimensi `d_model`; tiap fitur kategorikal memakai embedding lookup. Seluruh token digabung menjadi satu urutan (mirip pendekatan FT-Transformer untuk data tabular).

**Lightweight Transformer Encoder** — hanya 2–4 lapisan, `d_model` dan jumlah head kecil, agar ringan.

**Class-Specific Attention Pooling (cost-sensitive attention)** — alih-alih mean/CLS pooling biasa, setiap kelas memiliki *query* atensinya sendiri yang dipelajari (mirip Pooling-by-Multihead-Attention). Representasi khusus kelas ini memberi jalur langsung bagi gradien kelas minoritas untuk memengaruhi bobot atensi tanpa menambah parameter attention-head baru secara signifikan.

In [ ]:
class FeatureTokenizer(nn.Module):
    def __init__(self, n_numeric, cat_cardinalities, d_model):
        super().__init__()
        self.d_model = d_model
        self.num_proj = nn.Linear(1, d_model)
        self.num_bias = nn.Parameter(torch.zeros(n_numeric, d_model))
        self.cat_embeds = nn.ModuleList([nn.Embedding(card, d_model) for card in cat_cardinalities])
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls_token, std=0.02)

    def forward(self, x_num, x_cat):
        B, n_num = x_num.shape
        num_tok = self.num_proj(x_num.unsqueeze(-1)) + self.num_bias.unsqueeze(0)  # (B, n_num, d)
        cls_tok = self.cls_token.expand(B, -1, -1)
        token_list = [cls_tok, num_tok]
        if len(self.cat_embeds) > 0:  # sebagian dataset (mis. CICIDS2017) tidak punya fitur kategorikal
            cat_tok = torch.stack([emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeds)], dim=1)
            token_list.append(cat_tok)
        tokens = torch.cat(token_list, dim=1)  # (B, 1+n_num[+n_cat], d)
        return tokens


class ClassSpecificAttentionPooling(nn.Module):
    """Cost-sensitive attention: setiap kelas punya query sendiri untuk 'menarik'
    evidence dari sequence token yang paling relevan bagi kelas tersebut.

    PERBAIKAN dibanding versi awal:
    1. LayerNorm + residual pada representasi hasil pooling agar training stabil.
    2. Query per kelas di-normalisasi supaya skala antar kelas tidak timpang.
    """
    def __init__(self, d_model, n_classes, n_heads=4, dropout=0.1):
        super().__init__()
        self.n_classes = n_classes
        self.class_queries = nn.Parameter(torch.zeros(n_classes, d_model))
        nn.init.normal_(self.class_queries, std=0.02)
        self.q_norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.out_norm = nn.LayerNorm(d_model)

    def forward(self, tokens):
        B = tokens.size(0)
        q = self.q_norm(self.class_queries).unsqueeze(0).expand(B, -1, -1)  # (B, C, d)
        pooled, attn_weights = self.attn(q, tokens, tokens)                 # (B, C, d)
        pooled = self.out_norm(pooled + q)   # residual: jaga aliran gradien ke query kelas
        return pooled, attn_weights


class ClassWiseScoringHead(nn.Module):
    """Setiap kelas memiliki vektor skor sendiri (group fully-connected).

    Versi awal memakai satu nn.Linear(d_model, 1) yang DIBAGI seluruh kelas.
    Dengan bobot bersama, satu-satunya pembeda antar kelas hanyalah arah query,
    sehingga skor antar kelas mudah kolaps dan training menjadi tidak stabil.
    Di sini setiap kelas punya w_c dan b_c sendiri, mengikuti group-FC pada ML-Decoder.
    """
    def __init__(self, d_model, n_classes):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_classes, d_model))
        self.bias = nn.Parameter(torch.zeros(n_classes))
        nn.init.normal_(self.weight, std=d_model ** -0.5)

    def forward(self, pooled):                 # pooled: (B, C, d)
        return (pooled * self.weight.unsqueeze(0)).sum(-1) + self.bias   # (B, C)


class LightweightIDSTransformer(nn.Module):
    def __init__(self, n_numeric, cat_cardinalities, n_classes,
                 d_model=32, n_heads=4, n_layers=2, dim_ff=64, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_numeric, cat_cardinalities, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.pooling = ClassSpecificAttentionPooling(d_model, n_classes, n_heads=n_heads, dropout=dropout)
        self.class_head = ClassWiseScoringHead(d_model, n_classes)

    def forward(self, x_num, x_cat):
        tokens = self.tokenizer(x_num, x_cat)
        encoded = self.encoder(tokens)
        pooled, attn_w = self.pooling(encoded)   # (B, n_classes, d_model)
        logits = self.class_head(pooled)         # (B, n_classes)
        return logits, attn_w


class BaselineIDSTransformer(nn.Module):
    """Baseline: tokenizer sama, tapi mean-pooling + classifier standar (bukan cost-sensitive)."""
    def __init__(self, n_numeric, cat_cardinalities, n_classes,
                 d_model=32, n_heads=4, n_layers=2, dim_ff=64, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_numeric, cat_cardinalities, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, x_num, x_cat):
        tokens = self.tokenizer(x_num, x_cat)
        encoded = self.encoder(tokens)
        pooled = encoded.mean(dim=1)          # mean pooling biasa
        logits = self.classifier(pooled)
        return logits, None

## 5. Adaptive Focal Loss

Menggunakan skema **effective number of samples** (Cui et al., 2019) untuk menghitung bobot kelas `alpha` secara otomatis dari distribusi data latih, dikombinasikan dengan **focal modulating factor** `gamma` agar model lebih fokus pada sampel sulit — tanpa perlu augmentasi data sintetis.

In [ ]:
def compute_effective_num_weights(labels, n_classes, beta=0.9999):
    counts = np.bincount(labels, minlength=n_classes).astype(np.float64)
    counts = np.maximum(counts, 1)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / effective_num
    weights = weights / weights.sum() * n_classes
    return torch.tensor(weights, dtype=torch.float32)


class AdaptiveFocalLoss(nn.Module):
    def __init__(self, class_weights, gamma=2.0):
        super().__init__()
        self.register_buffer("alpha", class_weights)
        self.gamma = gamma

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1)
        probs = log_probs.exp()
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        alpha_t = self.alpha[targets]
        loss = -alpha_t * (1 - pt).pow(self.gamma) * log_pt
        return loss.mean()

class_weights = compute_effective_num_weights(y_train, n_classes).to(device)
print("Bobot kelas (effective number of samples):")
for name, w in zip(class_names, class_weights.tolist()):
    print(f"  {name:10s}: {w:.3f}")

## 6. Training & Evaluasi

In [ ]:
def evaluate(model, loader, class_names):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x_num, x_cat, y in loader:
            x_num, x_cat = x_num.to(device), x_cat.to(device)
            logits, _ = model(x_num, x_cat)
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(y.numpy())
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(class_names)), zero_division=0
    )
    macro_f1 = f1.mean()
    # G-mean = geometric mean of per-class recall
    recall_safe = np.clip(recall, 1e-6, 1.0)
    g_mean = np.exp(np.mean(np.log(recall_safe)))

    report_df = pd.DataFrame({
        "class": class_names, "precision": precision, "recall": recall,
        "f1": f1, "support": support,
    })
    return report_df, macro_f1, g_mean, y_true, y_pred


def train_model(model, loader_train, loader_test, loss_fn, epochs=None, lr=1e-3,
                verbose=True, log_every=0):
    """Latih model untuk jumlah epoch TETAP.

    PERUBAHAN PENTING dibanding versi sebelumnya:

    1. TIDAK ADA lagi seleksi bobot "epoch terbaik" berdasarkan test set.
       Versi lama mengevaluasi test set setiap epoch lalu menyimpan bobot dengan
       macro-F1 tertinggi. Itu adalah pemilihan model di atas data uji — bentuk
       kebocoran (leakage) yang akan disorot reviewer, karena angka yang
       dilaporkan menjadi optimistis. Sekarang model dilatih sejumlah epoch
       tetap dan dievaluasi SEKALI di akhir.

    2. Tidak ada evaluasi test set per epoch. Selain menghilangkan leakage, ini
       memangkas waktu komputasi secara besar: pada 10 seed x 4 varian,
       evaluasi per epoch sebelumnya mendominasi total runtime.

    log_every > 0 hanya untuk pemantauan: mencetak loss tiap n epoch
    (loss data latih saja, tidak menyentuh test set).
    """
    epochs = globals().get("EPOCHS", 15) if epochs is None else epochs
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = max(1, epochs * len(loader_train))
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=steps, pct_start=0.2)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for x_num, x_cat, y in loader_train:
            x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
            optimizer.zero_grad()
            logits, _ = model(x_num, x_cat)
            loss = loss_fn(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            sched.step()
            total_loss += loss.item() * x_num.size(0)

        if verbose and log_every and (epoch % log_every == 0 or epoch == epochs):
            print(f"  epoch {epoch:2d}/{epochs} | train loss={total_loss/len(loader_train.dataset):.4f}")

    return model


In [ ]:
# PERBAIKAN: d_model dinaikkan (32 terlalu sempit untuk menampung C query kelas terpisah;
# dengan 4 head, tiap head hanya mendapat 8 dimensi). Epoch ditambah karena OneCycleLR
# membutuhkan lebih banyak langkah untuk konvergen.
D_MODEL, N_HEADS, N_LAYERS, EPOCHS = 64, 4, 2, 15

print("=== Melatih model yang DIUSULKAN (Cost-Sensitive Attention + Adaptive Focal Loss) ===")
proposed_model = LightweightIDSTransformer(
    n_numeric, cat_cardinalities, n_classes,
    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
)
focal_loss = AdaptiveFocalLoss(class_weights, gamma=2.0)
proposed_model = train_model(proposed_model, train_loader, test_loader, focal_loss, epochs=EPOCHS, log_every=5)

In [ ]:
print("=== Melatih model BASELINE (mean-pooling + cross-entropy standar) ===")
baseline_model = BaselineIDSTransformer(
    n_numeric, cat_cardinalities, n_classes,
    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
)
ce_loss = nn.CrossEntropyLoss()
baseline_model = train_model(baseline_model, train_loader, test_loader, ce_loss, epochs=EPOCHS, log_every=5)

## 7. Perbandingan Hasil (fokus otomatis ke kelas paling minoritas)

In [ ]:
report_proposed, f1_proposed, gmean_proposed, yt, yp_prop = evaluate(proposed_model, test_loader, class_names)
report_baseline, f1_baseline, gmean_baseline, _, yp_base = evaluate(baseline_model, test_loader, class_names)

print("--- Model yang Diusulkan ---")
display(report_proposed)
print(f"Macro-F1: {f1_proposed:.4f} | G-mean: {gmean_proposed:.4f}\n")

print("--- Baseline ---")
display(report_baseline)
print(f"Macro-F1: {f1_baseline:.4f} | G-mean: {gmean_baseline:.4f}")

In [ ]:
# Tentukan otomatis 2 kelas paling minoritas berdasarkan jumlah sampel di data latih
# (berlaku untuk dataset manapun: R2L/U2R pada NSL-KDD, Heartbleed/Infiltration pada CICIDS2017, dst.)
minority_idx = pd.Series(y_train).value_counts().sort_values().index[:2]
minority_names = [class_names[i] for i in minority_idx]
print("Kelas minoritas yang menjadi fokus evaluasi:", minority_names)

def get_f1(report_df, cls_name):
    row = report_df.loc[report_df["class"] == cls_name, "f1"]
    return row.values[0] if len(row) else None

comparison = pd.DataFrame({
    "Model": ["Proposed (Cost-Sensitive Attn + Focal)", "Baseline (Mean-pool + CE)"],
    "Macro-F1": [f1_proposed, f1_baseline],
    "G-mean": [gmean_proposed, gmean_baseline],
    f"F1_{minority_names[0]}": [get_f1(report_proposed, minority_names[0]), get_f1(report_baseline, minority_names[0])],
    f"F1_{minority_names[1]}": [get_f1(report_proposed, minority_names[1]), get_f1(report_baseline, minority_names[1])],
})
comparison

## 8. Profil Efisiensi Komputasi

Mengukur empat dimensi efisiensi yang lazim dilaporkan di paper IDS:
1. **Jumlah parameter** dan **ukuran model** (MB)
2. **FLOPs** per sampel (via `ptflops`; jika gagal, dihitung analitis sebagai fallback)
3. **Latensi single-thread** (`torch.set_num_threads(1)`) — mensimulasikan kondisi CPU terbatas
4. **Dynamic quantization** (INT8) — seberapa jauh model bisa dikecilkan tanpa kehilangan akurasi

> Sekali lagi: ini **simulasi resource-constrained**, bukan pengujian edge device. Laporkan apa adanya di paper.

In [ ]:
# Instalasi tanpa --upgrade agar tidak mengubah versi paket bawaan Colab.
!pip -q install ptflops 2>/dev/null || echo "Instalasi ptflops gagal - sel ini akan dilewati."

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def model_size_mb(model):
    total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    return total_bytes / (1024 ** 2)

def measure_latency(model, loader, n_batches=30, single_thread=True):
    """Latensi inferensi di CPU. single_thread=True mensimulasikan perangkat terbatas."""
    prev_threads = torch.get_num_threads()
    if single_thread:
        torch.set_num_threads(1)
    model = model.to("cpu")
    model.eval()
    times, n_samples = [], 0
    it = iter(loader)
    # warm-up agar pengukuran tidak terpengaruh inisialisasi lazy
    with torch.no_grad():
        x_num, x_cat, _ = next(it)
        _ = model(x_num.to("cpu"), x_cat.to("cpu"))
    with torch.no_grad():
        for _ in range(n_batches):
            try:
                x_num, x_cat, _ = next(it)
            except StopIteration:
                it = iter(loader)
                x_num, x_cat, _ = next(it)
            x_num, x_cat = x_num.to("cpu"), x_cat.to("cpu")
            start = time.perf_counter()
            _ = model(x_num, x_cat)
            times.append((time.perf_counter() - start) * 1000)
            n_samples += x_num.size(0)
    torch.set_num_threads(prev_threads)
    per_batch = np.mean(times)
    per_sample = sum(times) / n_samples
    return per_batch, np.std(times), per_sample

def estimate_flops(model, n_numeric, n_cat, d_model, n_layers, n_classes):
    """Estimasi FLOPs per sampel. Coba ptflops dulu; jika gagal, hitung analitis."""
    try:
        from ptflops import get_model_complexity_info

        class Wrapper(nn.Module):
            def __init__(self, m, n_cat_):
                super().__init__()
                self.m = m
                self.n_cat = n_cat_
            def forward(self, x):
                x_cat = torch.zeros((x.size(0), self.n_cat), dtype=torch.long, device=x.device)
                out, _ = self.m(x, x_cat)
                return out

        wrapped = Wrapper(model, n_cat).to("cpu").eval()
        macs, _ = get_model_complexity_info(
            wrapped, (n_numeric,), as_strings=False,
            print_per_layer_stat=False, verbose=False,
        )
        return float(macs) * 2, "ptflops"
    except Exception as e:
        # Fallback analitis: attention + FFN untuk seq_len token
        seq_len = 1 + n_numeric + n_cat
        per_layer = (
            4 * seq_len * d_model * d_model      # proyeksi Q,K,V,O
            + 2 * seq_len * seq_len * d_model    # QK^T dan attn@V
            + 2 * seq_len * d_model * (2 * d_model)  # FFN (dim_ff = 2*d_model)
        )
        total = n_layers * per_layer
        total += n_classes * seq_len * d_model   # class-specific attention pooling
        return float(total) * 2, "analitis (fallback)"

rows = []
for name, m in [("Proposed", proposed_model), ("Baseline", baseline_model)]:
    n_params = count_params(m)
    size_mb = model_size_mb(m)
    lat_batch, lat_std, lat_sample = measure_latency(m, test_loader, single_thread=True)
    flops, flops_src = estimate_flops(m, n_numeric, n_categorical, D_MODEL, N_LAYERS, n_classes)
    rows.append({
        "Model": name,
        "Params": n_params,
        "Size (MB)": round(size_mb, 4),
        "FLOPs/sample": f"{flops:,.0f}",
        "FLOPs src": flops_src,
        "Latency/batch (ms)": round(lat_batch, 3),
        "Latency/sample (ms)": round(lat_sample, 5),
    })

efficiency_df = pd.DataFrame(rows)
display(efficiency_df)

proposed_model.to(device); baseline_model.to(device);

In [ ]:
import copy, io

def evaluate_cpu(model, loader, class_names):
    """Versi evaluate() yang dipaksa berjalan di CPU (model terkuantisasi hanya jalan di CPU)."""
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x_num, x_cat, y in loader:
            logits, _ = model(x_num.to("cpu"), x_cat.to("cpu"))
            all_preds.append(logits.argmax(dim=-1).cpu().numpy())
            all_targets.append(y.numpy())
    y_pred = np.concatenate(all_preds); y_true = np.concatenate(all_targets)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(class_names)), zero_division=0
    )
    recall_safe = np.clip(recall, 1e-6, 1.0)
    return f1.mean(), float(np.exp(np.mean(np.log(recall_safe))))


def state_dict_size_mb(model):
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / (1024 ** 2)


# ---- Dynamic quantization (INT8) pada layer Linear ----
# PERBAIKAN BUG: versi awal memanggil quantize_dynamic(model, {nn.Linear}) dan gagal dengan
# "'function' object has no attribute 'device'". Penyebabnya nn.MultiheadAttention memuat
# NonDynamicallyQuantizableLinear (subclass nn.Linear) yang memang tidak boleh dikuantisasi
# lewat jalur ini. Solusinya: kuantisasi HANYA Linear yang berada di luar MultiheadAttention,
# dengan menyebut nama modulnya secara eksplisit.
fp32_model = copy.deepcopy(proposed_model).to("cpu").eval()

def linear_names_outside_mha(model):
    """Nama modul nn.Linear yang tidak berada di dalam MultiheadAttention."""
    mha_prefixes = [n for n, m in model.named_modules()
                    if isinstance(m, nn.MultiheadAttention)]
    names = []
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and not any(
            n == p or n.startswith(p + ".") for p in mha_prefixes
        ):
            names.append(n)
    return names

f1_int8 = gmean_int8 = size_int8 = lat_int8 = None
quantization_succeeded = False
try:
    from torch.ao.quantization import default_dynamic_qconfig, quantize_dynamic
    target = linear_names_outside_mha(fp32_model)
    print(f"Layer Linear yang dikuantisasi ({len(target)}):", target)
    qspec = {name: default_dynamic_qconfig for name in target}
    int8_model = quantize_dynamic(fp32_model, qconfig_spec=qspec, dtype=torch.qint8)

    f1_int8, gmean_int8 = evaluate_cpu(int8_model, test_loader, class_names)
    size_int8 = state_dict_size_mb(int8_model)
    lat_int8, _, _ = measure_latency(int8_model, test_loader, single_thread=True)
    quantization_succeeded = True
    print("Dynamic quantization berhasil.")
except Exception as e:
    print(f"Dynamic quantization gagal: {type(e).__name__}: {e}")

# Evaluasi FP32 sebagai pembanding
f1_fp32, gmean_fp32 = evaluate_cpu(fp32_model, test_loader, class_names)
size_fp32 = state_dict_size_mb(fp32_model)
lat_fp32, _, _ = measure_latency(fp32_model, test_loader, single_thread=True)

# PENTING: bila kuantisasi gagal, baris INT8 diisi NaN - JANGAN menyalin nilai FP32.
# Versi awal menyalin nilai FP32 ke baris INT8, sehingga tabel terbaca seolah kuantisasi
# tidak menurunkan performa sama sekali. Itu menyesatkan bila dilaporkan di paper.
quant_df = pd.DataFrame({
    "Precision": ["FP32", "INT8 (dynamic quant)"],
    "Size on disk (MB)": [round(size_fp32, 4), round(size_int8, 4) if quantization_succeeded else np.nan],
    "Macro-F1": [round(f1_fp32, 4), round(f1_int8, 4) if quantization_succeeded else np.nan],
    "G-mean": [round(gmean_fp32, 4), round(gmean_int8, 4) if quantization_succeeded else np.nan],
    "Latency/batch (ms)": [round(lat_fp32, 3), round(lat_int8, 3) if quantization_succeeded else np.nan],
})
display(quant_df)

if quantization_succeeded:
    drop = f1_fp32 - f1_int8
    print(f"Selisih macro-F1 FP32 -> INT8: {drop:+.4f}. "
          "Selisih kecil menunjukkan model tetap robust setelah dikompresi.")
else:
    print("Kuantisasi gagal, sehingga baris INT8 sengaja dikosongkan (NaN). "
          "JANGAN melaporkan angka INT8 di paper sebelum sel ini berhasil dijalankan.")

proposed_model.to(device);


## 9. Studi Ablasi (multi-seed)

**Ini sel paling menentukan untuk paper.** Ablasi memisahkan kontribusi tiap komponen:

| Varian | Pooling | Loss |
|---|---|---|
| Full | class-specific attention | adaptive focal |
| Attn-only | class-specific attention | cross-entropy |
| Focal-only | mean-pooling | adaptive focal |
| Baseline | mean-pooling | cross-entropy |

Pada percobaan sebelumnya ablasi hanya dijalankan **satu seed**, sehingga selisih
Full vs Focal-only tidak dapat dibedakan dari noise — simpangan baku antar seed
ternyata seukuran selisihnya sendiri.

Di sini setiap varian dijalankan pada beberapa seed dan dilaporkan `mean ± std`,
lalu diuji dengan Wilcoxon. Hanya dengan cara ini pertanyaan intinya bisa dijawab:
**apakah class-specific attention pooling benar-benar menyumbang sesuatu di atas focal loss?**


In [ ]:
from scipy.stats import wilcoxon

# 10 seed. Pada n=6, p-value terkecil Wilcoxon dua sisi adalah 0.031 - persis di ambang
# 0.05, sehingga TIDAK ada ruang untuk koreksi multiple comparison. Pada n=10, p minimum
# turun ke 0.002 sehingga hasil masih dapat bertahan setelah koreksi Holm.
ABL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

VARIANTS = {
    "Full (Attn+Focal)":   ("proposed", "focal"),
    "Attn-only":           ("proposed", "ce"),
    "Focal-only":          ("baseline", "focal"),
    "Baseline":            ("baseline", "ce"),
}

def build_and_train(model_type, loss_type, seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    cls = LightweightIDSTransformer if model_type == "proposed" else BaselineIDSTransformer
    m = cls(n_numeric, cat_cardinalities, n_classes,
            d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS)
    loss_fn = (AdaptiveFocalLoss(class_weights, gamma=2.0) if loss_type == "focal"
               else nn.CrossEntropyLoss())
    m = train_model(m, train_loader, test_loader, loss_fn, verbose=False)
    rep, mf1, gm, _, _ = evaluate(m, test_loader, class_names)
    f1_min = rep.loc[rep["class"] == minority_names[0], "f1"].values[0]
    f1_min2 = rep.loc[rep["class"] == minority_names[1], "f1"].values[0]
    return mf1, gm, f1_min, f1_min2

abl_raw = {v: [] for v in VARIANTS}
for seed in ABL_SEEDS:
    print(f"--- Seed {seed} ---")
    for vname, (mt, lt) in VARIANTS.items():
        res = build_and_train(mt, lt, seed)
        abl_raw[vname].append(res)
        print(f"  {vname:20s} macroF1={res[0]:.4f}  gmean={res[1]:.4f}  "
              f"F1({minority_names[0]})={res[2]:.4f}  F1({minority_names[1]})={res[3]:.4f}")
    # Simpan setelah SETIAP seed: bila runtime terputus, progres tidak hilang total
    pd.DataFrame([{"seed": sd, "varian": v, "macro_f1": r[0], "g_mean": r[1],
                   f"f1_{minority_names[0]}": r[2], f"f1_{minority_names[1]}": r[3]}
                  for v, rs in abl_raw.items()
                  for sd, r in zip(ABL_SEEDS, rs)]).to_csv(f"ablasi_partial_{TAG}.csv", index=False)

abl_arr = {v: np.array(r) for v, r in abl_raw.items()}
METRICS = ["Macro-F1", "G-mean", f"F1 ({minority_names[0]})", f"F1 ({minority_names[1]})"]

ablation_stats = pd.DataFrame({
    "Varian": list(VARIANTS),
    **{m: [f"{abl_arr[v][:, i].mean():.4f} ± {abl_arr[v][:, i].std():.4f}" for v in VARIANTS]
       for i, m in enumerate(METRICS)}
})
display(ablation_stats)

In [ ]:
# =====================================================================
# Uji signifikansi + KOREKSI MULTIPLE COMPARISON (Holm-Bonferroni)
# =====================================================================
# Sel ini menjalankan 3 perbandingan x 4 metrik = 12 uji sekaligus.
# Tanpa koreksi, peluang memperoleh minimal satu p < 0.05 secara kebetulan
# adalah 1 - 0.95^12 = 46 persen. Reviewer statistik akan menanyakan ini,
# sehingga koreksi dilaporkan sejak awal.
#
# Holm-Bonferroni dipilih dibandingkan Bonferroni murni karena lebih kuat
# (menolak lebih banyak hipotesis null yang memang salah) namun tetap
# mengendalikan family-wise error rate pada taraf yang sama.

print(f"Wilcoxon signed-rank test, berpasangan per seed (n = {len(ABL_SEEDS)})")
print(f"p minimum yang dapat dicapai pada n={len(ABL_SEEDS)}: "
      f"{2/(2**len(ABL_SEEDS)):.4f}\n")

PAIRS = [
    ("Full (Attn+Focal)", "Focal-only", "kontribusi ATTENTION di atas focal loss"),
    ("Attn-only",         "Baseline",   "kontribusi ATTENTION sendirian"),
    ("Focal-only",        "Baseline",   "kontribusi FOCAL LOSS sendirian"),
]

# ---- Kumpulkan seluruh uji terlebih dahulu ----
tests = []
for a_name, b_name, tujuan in PAIRS:
    A, B = abl_arr[a_name], abl_arr[b_name]
    for i, mname in enumerate(METRICS):
        a, b = A[:, i], B[:, i]
        delta = float(a.mean() - b.mean())
        if np.allclose(a, b):
            tests.append({"pair": f"{a_name} vs {b_name}", "tujuan": tujuan,
                          "metric": mname, "delta": delta, "p": np.nan})
            continue
        try:
            _, pval = wilcoxon(a, b)
        except ValueError:
            pval = np.nan
        tests.append({"pair": f"{a_name} vs {b_name}", "tujuan": tujuan,
                      "metric": mname, "delta": delta, "p": float(pval)})

# ---- Koreksi Holm-Bonferroni ----
def holm_bonferroni(pvals, alpha=0.05):
    """Kembalikan (p_terkoreksi, tolak_H0). NaN dilewati dari perhitungan."""
    idx = [i for i, p in enumerate(pvals) if not np.isnan(p)]
    m = len(idx)
    order = sorted(idx, key=lambda i: pvals[i])
    p_adj = [np.nan] * len(pvals)
    running = 0.0
    for rank, i in enumerate(order):
        val = (m - rank) * pvals[i]
        running = max(running, val)          # jaga sifat monoton
        p_adj[i] = min(1.0, running)
    reject = [(not np.isnan(p_adj[i])) and p_adj[i] < alpha for i in range(len(pvals))]
    return p_adj, reject

p_raw = [t["p"] for t in tests]
p_adj, reject = holm_bonferroni(p_raw)
for t, pa, rj in zip(tests, p_adj, reject):
    t["p_holm"] = pa
    t["signifikan"] = rj

# ---- Tampilkan ----
holm_df = pd.DataFrame([{
    "Perbandingan": t["pair"],
    "Metrik": t["metric"],
    "Delta": round(t["delta"], 4),
    "p (mentah)": round(t["p"], 4) if not np.isnan(t["p"]) else None,
    "p (Holm)": round(t["p_holm"], 4) if not np.isnan(t["p_holm"]) else None,
    "Signifikan": "ya" if t["signifikan"] else "tidak",
} for t in tests])
display(holm_df)

n_raw = sum(1 for t in tests if not np.isnan(t["p"]) and t["p"] < 0.05)
n_holm = sum(1 for t in tests if t["signifikan"])
print(f"\nSignifikan sebelum koreksi : {n_raw} dari {len(tests)} uji")
print(f"Signifikan setelah Holm    : {n_holm} dari {len(tests)} uji")

print("\nCara membaca:")
print("  - Kolom 'p (Holm)' inilah yang dilaporkan di paper, bukan p mentah.")
print("  - Baris 'Full vs Focal-only' adalah penentu klaim utama: bila tidak")
print("    signifikan, attention pooling tidak menyumbang di atas focal loss.")
print("  - Bila SELURUH baris menjadi tidak signifikan setelah koreksi, itu")
print("    tetap temuan yang sah dan harus dilaporkan apa adanya: selisih antar")
print("    strategi lebih kecil daripada yang diasumsikan literatur.")

## 10. Uji Signifikansi Statistik (Multi-Seed)

Melatih ulang model yang diusulkan dan baseline dengan **beberapa seed berbeda**, lalu melaporkan
`mean ± std` dan menjalankan **Wilcoxon signed-rank test** (uji non-parametrik berpasangan).

Ini menutup salah satu kelemahan paling umum pada paper IDS: melaporkan satu angka tunggal
tanpa variansi, sehingga reviewer tidak bisa menilai apakah perbedaan performa itu nyata atau kebetulan.

> `N_SEEDS = 10`. Dengan n = 5, p-value terkecil yang dapat dicapai Wilcoxon dua sisi
> adalah 0.0625, sehingga klaim `p < 0.05` mustahil secara matematis. Pada n = 10,
> p minimum menjadi 0.002 — cukup untuk bertahan terhadap koreksi Holm-Bonferroni.

In [ ]:
from scipy.stats import wilcoxon

# PERBAIKAN: n=5 membuat p<0.05 mustahil pada Wilcoxon dua sisi (p minimum 0.0625).
N_SEEDS = 10
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51][:N_SEEDS]

def run_single_seed(seed, model_type, loss_type):
    """Latih satu model dengan seed tertentu, kembalikan (macro_f1, g_mean, minority_f1).

    PERBAIKAN: sebelumnya sel ini menyalin loop training sendiri tanpa LR scheduler,
    gradient clipping, maupun pemilihan epoch terbaik - sehingga hasilnya TIDAK sebanding
    dengan model yang dilatih train_model() di Bagian 6. Sekarang memakai train_model()
    yang sama persis.
    """
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    cls = LightweightIDSTransformer if model_type == "proposed" else BaselineIDSTransformer
    m = cls(n_numeric, cat_cardinalities, n_classes,
            d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS)
    loss_fn = (AdaptiveFocalLoss(class_weights, gamma=2.0) if loss_type == "focal"
               else nn.CrossEntropyLoss())

    m = train_model(m, train_loader, test_loader, loss_fn, verbose=False)

    rep, mf1, gm, _, _ = evaluate(m, test_loader, class_names)
    min_f1 = rep.loc[rep["class"] == minority_names[0], "f1"].values[0]
    return mf1, gm, min_f1

results = {"proposed": [], "baseline": []}
for seed in SEEDS:
    print(f"--- Seed {seed} ---")
    mf1_p, gm_p, minf1_p = run_single_seed(seed, "proposed", "focal")
    mf1_b, gm_b, minf1_b = run_single_seed(seed, "baseline", "ce")
    results["proposed"].append((mf1_p, gm_p, minf1_p))
    results["baseline"].append((mf1_b, gm_b, minf1_b))
    print(f"  proposed: macroF1={mf1_p:.4f} gmean={gm_p:.4f} | baseline: macroF1={mf1_b:.4f} gmean={gm_b:.4f}")

prop_arr = np.array(results["proposed"])
base_arr = np.array(results["baseline"])

stats_df = pd.DataFrame({
    "Metrik": ["Macro-F1", "G-mean", f"F1 ({minority_names[0]})"],
    "Proposed (mean ± std)": [f"{prop_arr[:, i].mean():.4f} ± {prop_arr[:, i].std():.4f}" for i in range(3)],
    "Baseline (mean ± std)": [f"{base_arr[:, i].mean():.4f} ± {base_arr[:, i].std():.4f}" for i in range(3)],
})
display(stats_df)

In [ ]:
# Wilcoxon + koreksi Holm untuk perbandingan utama (proposed vs baseline)
print(f"Wilcoxon signed-rank test (n = {N_SEEDS} seed berpasangan)\n")
metric_names = ["Macro-F1", "G-mean", f"F1 ({minority_names[0]})"]

rows_main = []
for i, mname in enumerate(metric_names):
    a, b = prop_arr[:, i], base_arr[:, i]
    delta = float(a.mean() - b.mean())
    if np.allclose(a, b):
        rows_main.append({"Metrik": mname, "Delta": round(delta, 4), "p": np.nan})
        continue
    try:
        _, pv = wilcoxon(a, b)
    except ValueError:
        pv = np.nan
    rows_main.append({"Metrik": mname, "Delta": round(delta, 4), "p": float(pv)})

p_adj_main, reject_main = holm_bonferroni([r["p"] for r in rows_main])
for r, pa, rj in zip(rows_main, p_adj_main, reject_main):
    r["p (Holm)"] = round(pa, 4) if not np.isnan(pa) else None
    r["p"] = round(r["p"], 4) if not np.isnan(r["p"]) else None
    r["Signifikan"] = "ya" if rj else "tidak"

display(pd.DataFrame(rows_main))

print(f"\nCatatan: koreksi Holm di sini hanya mencakup {len(rows_main)} uji pada sel ini.")
print("Bila hasil sel ini dan sel ablasi dilaporkan bersama dalam satu tabel di paper,")
print("koreksi harus dihitung ulang atas GABUNGAN seluruh uji, bukan per sel.")

## 11. Perbandingan dengan SMOTE (Baseline Penyeimbangan Konvensional)

Untuk mendukung klaim *"efektif tanpa augmentasi data yang mahal"*, model yang diusulkan perlu
dibandingkan langsung dengan pendekatan oversampling konvensional. Di sini SMOTE diterapkan pada
data latih, lalu model baseline (mean-pooling + cross-entropy) dilatih di atasnya.

Yang dilaporkan bukan hanya performanya, tetapi juga **biaya komputasi tambahan** dari SMOTE
(waktu resampling dan pembengkakan jumlah baris data latih).

In [ ]:
# Instalasi tanpa --upgrade agar tidak mengubah versi paket bawaan Colab.
!pip -q install imbalanced-learn 2>/dev/null || echo "Instalasi imbalanced-learn gagal - sel ini akan dilewati."
from imblearn.over_sampling import SMOTE

# SMOTE hanya bekerja pada fitur numerik kontinu. Untuk NSL-KDD, fitur kategorikal
# disertakan sebagai nilai numerik terenkode — ini penyederhanaan yang perlu dicatat di paper
# (SMOTE-NC adalah pilihan yang lebih tepat untuk fitur campuran).
X_train_combined = np.hstack([X_train_num, X_train_cat]) if n_categorical > 0 else X_train_num

smote_start = time.perf_counter()
min_count = pd.Series(y_train).value_counts().min()
k_neighbors = min(5, max(1, min_count - 1))
try:
    smote = SMOTE(random_state=SEED, k_neighbors=k_neighbors)
    X_res, y_res = smote.fit_resample(X_train_combined, y_train)
    smote_time = time.perf_counter() - smote_start
    print(f"SMOTE selesai dalam {smote_time:.2f} detik")
    print(f"Baris data latih: {len(y_train):,} -> {len(y_res):,} "
          f"({len(y_res)/len(y_train):.2f}x lebih besar)")
    smote_ok = True
except ValueError as e:
    print("SMOTE gagal:", e)
    smote_ok = False

In [ ]:
if smote_ok:
    # Pisahkan kembali bagian numerik dan kategorikal setelah resampling
    if n_categorical > 0:
        X_res_num = X_res[:, :X_train_num.shape[1]]
        X_res_cat = np.rint(X_res[:, X_train_num.shape[1]:]).astype(np.int64)
        # jaga agar indeks kategorikal hasil interpolasi tetap dalam rentang embedding yang valid
        for j, card in enumerate(cat_cardinalities):
            X_res_cat[:, j] = np.clip(X_res_cat[:, j], 0, card - 1)
    else:
        X_res_num = X_res
        X_res_cat = np.zeros((len(y_res), 0), dtype=np.int64)

    smote_ds = IDSDataset(X_res_num, X_res_cat, y_res)
    smote_loader = torch.utils.data.DataLoader(smote_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    print("=== Melatih baseline + SMOTE ===")
    torch.manual_seed(SEED); np.random.seed(SEED)
    smote_model = BaselineIDSTransformer(n_numeric, cat_cardinalities, n_classes,
                                         d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS)
    train_start = time.perf_counter()
    smote_model = train_model(smote_model, smote_loader, test_loader, nn.CrossEntropyLoss(), epochs=EPOCHS)
    smote_train_time = time.perf_counter() - train_start

    rep_smote, f1_smote, gmean_smote, _, _ = evaluate(smote_model, test_loader, class_names)
    display(rep_smote)

    smote_comparison = pd.DataFrame({
        "Pendekatan": [
            "Proposed (Cost-Sensitive Attn + Focal, tanpa augmentasi)",
            "Baseline + SMOTE",
            "Baseline (tanpa apa-apa)",
        ],
        "Macro-F1": [f1_proposed, f1_smote, f1_baseline],
        "G-mean": [gmean_proposed, gmean_smote, gmean_baseline],
        "Baris data latih": [f"{len(y_train):,}", f"{len(y_res):,}", f"{len(y_train):,}"],
        "Overhead preprocessing (s)": [0.0, round(smote_time, 2), 0.0],
    })
    display(smote_comparison)
    print("\nInilah bukti kuantitatif untuk klaim efisiensi: bandingkan kolom "
          "'Baris data latih' dan 'Overhead preprocessing' terhadap selisih Macro-F1 / G-mean.")

## 12. Analisis Attention (XAI)

Salah satu keunggulan `ClassSpecificAttentionPooling` adalah setiap kelas memiliki *query* sendiri,
sehingga bobot atensinya dapat dibaca langsung: **fitur mana yang paling diperhatikan model
ketika menilai suatu kelas tertentu**.

Analisis ini menambahkan dimensi interpretability yang banyak diminta reviewer, dan seluruhnya
berbasis software — tidak butuh perangkat tambahan.

In [ ]:
import matplotlib.pyplot as plt

# Nama token sesuai urutan yang dibentuk FeatureTokenizer: [CLS] + numerik + kategorikal
token_names = ["[CLS]"] + list(numeric_cols) + list(categorical_cols)

proposed_model.eval()
attn_accum = None
n_batches_xai = 0
with torch.no_grad():
    for x_num, x_cat, y in test_loader:
        x_num, x_cat = x_num.to(device), x_cat.to(device)
        _, attn_w = proposed_model(x_num, x_cat)   # (B, n_classes, seq_len)
        batch_mean = attn_w.mean(dim=0).cpu().numpy()
        attn_accum = batch_mean if attn_accum is None else attn_accum + batch_mean
        n_batches_xai += 1
        if n_batches_xai >= 20:
            break

attn_mean = attn_accum / n_batches_xai   # (n_classes, seq_len)
print("Bentuk matriks atensi rata-rata:", attn_mean.shape)

TOP_K = 10
for ci, cname in enumerate(class_names):
    order = np.argsort(attn_mean[ci])[::-1][:TOP_K]
    tops = [(token_names[j] if j < len(token_names) else f"tok_{j}", attn_mean[ci, j]) for j in order]
    print(f"\n[{cname}] top-{TOP_K} fitur berdasarkan bobot atensi:")
    for fname, w in tops:
        print(f"    {fname:35s} {w:.4f}")

In [ ]:
# Heatmap: kelas x top fitur (agregat)
top_overall = np.argsort(attn_mean.mean(axis=0))[::-1][:15]
labels_top = [token_names[j] if j < len(token_names) else f"tok_{j}" for j in top_overall]

fig, ax = plt.subplots(figsize=(12, 0.6 * len(class_names) + 2))
im = ax.imshow(attn_mean[:, top_overall], aspect="auto", cmap="viridis")
ax.set_xticks(range(len(labels_top)))
ax.set_xticklabels(labels_top, rotation=45, ha="right")
ax.set_yticks(range(len(class_names)))
ax.set_yticklabels(class_names)
ax.set_title("Bobot Class-Specific Attention (rata-rata pada data uji)")
fig.colorbar(im, ax=ax, label="bobot atensi")
plt.tight_layout()
plt.savefig(f'fig2_attention_heatmap_{TAG}.png', dpi=300, bbox_inches='tight',
            facecolor='white')
print(f'Tersimpan: fig2_attention_heatmap_{TAG}.png  (untuk Gambar 2 di naskah)')
plt.show()

print("Interpretasi untuk paper: bandingkan baris kelas minoritas dengan kelas mayoritas. "
      "Jika query kelas minoritas memusat pada fitur yang berbeda dari kelas mayoritas, "
      "itu bukti kualitatif bahwa mekanisme cost-sensitive attention benar-benar bekerja, "
      "bukan sekadar menaikkan angka metrik.")

## 13. Baseline Ensemble Klasik (Random Forest, XGBoost, Voting)

Reviewer pada bidang IDS hampir selalu menanyakan: *"mengapa memakai Transformer bila tree ensemble
yang jauh lebih murah sudah cukup baik?"* Bagian ini menjawabnya secara empiris.

Tiga pembanding ditambahkan:
1. **Random Forest** — pada berbagai systematic review IDS, RF konsisten menjadi baseline terkuat.
2. **XGBoost** — gradient boosting yang umum dipakai pada data tabular.
3. **Soft-voting ensemble** — gabungan RF + XGBoost.

Ketiganya juga menggunakan `class_weight` / `sample_weight` agar perbandingannya adil: model yang
diusulkan memakai cost-sensitive learning, sehingga pembandingnya pun harus diberi perlakuan setara.

Selain performa, dicatat pula **biaya komputasi**: waktu latih, waktu inferensi, dan ukuran model.
Inilah argumen inti — apakah selisih performa sepadan dengan selisih biayanya.

In [ ]:
# Instalasi tanpa --upgrade agar tidak mengubah versi paket bawaan Colab.
!pip -q install xgboost 2>/dev/null || echo "Instalasi xgboost gagal - sel ini akan dilewati."
import pickle, io
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

# Tree ensemble bekerja pada matriks fitur datar (numerik + kategorikal terenkode)
X_tr = np.hstack([X_train_num, X_train_cat]) if n_categorical > 0 else X_train_num
X_te = np.hstack([X_test_num, X_test_cat]) if n_categorical > 0 else X_test_num
print("Bentuk matriks fitur:", X_tr.shape, "->", X_te.shape)

def eval_sklearn(model, X, y_true, class_names):
    y_pred = model.predict(X)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(class_names)), zero_division=0
    )
    recall_safe = np.clip(recall, 1e-6, 1.0)
    g_mean = float(np.exp(np.mean(np.log(recall_safe))))
    rep = pd.DataFrame({"class": class_names, "precision": precision,
                        "recall": recall, "f1": f1, "support": support})
    return rep, f1.mean(), g_mean

def sklearn_model_size_mb(model):
    buf = io.BytesIO()
    pickle.dump(model, buf)
    return buf.getbuffer().nbytes / (1024 ** 2)

def measure_sklearn_latency(model, X, n_repeat=5):
    """Latensi inferensi per batch, ukuran batch disamakan dengan BATCH_SIZE model deep learning."""
    Xb = X[:BATCH_SIZE]
    _ = model.predict(Xb)  # warm-up
    times = []
    for _ in range(n_repeat):
        start = time.perf_counter()
        _ = model.predict(Xb)
        times.append((time.perf_counter() - start) * 1000)
    return float(np.mean(times))

In [ ]:
ensemble_results = {}

# ---------- Random Forest ----------
print("=== Melatih Random Forest ===")
t0 = time.perf_counter()
rf = RandomForestClassifier(
    n_estimators=100, max_depth=None, n_jobs=-1,
    class_weight="balanced",   # padanan cost-sensitive learning
    random_state=SEED,
)
rf.fit(X_tr, y_train)
rf_train_time = time.perf_counter() - t0
rep_rf, f1_rf, gmean_rf = eval_sklearn(rf, X_te, y_test, class_names)
print(f"Waktu latih: {rf_train_time:.2f}s | macro-F1={f1_rf:.4f} | G-mean={gmean_rf:.4f}")
display(rep_rf)
ensemble_results["Random Forest"] = {
    "macro_f1": f1_rf, "g_mean": gmean_rf, "train_time": rf_train_time,
    "size_mb": sklearn_model_size_mb(rf), "latency_ms": measure_sklearn_latency(rf, X_te),
    "report": rep_rf,
}

In [ ]:
# ---------- XGBoost ----------
print("=== Melatih XGBoost ===")
# sample_weight sebagai padanan class-weighting pada XGBoost multiclass
cls_counts = np.bincount(y_train, minlength=n_classes).astype(float)
cls_w = len(y_train) / (n_classes * np.maximum(cls_counts, 1))
sample_w = cls_w[y_train]

t0 = time.perf_counter()
xgb = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.3,
    objective="multi:softprob", num_class=n_classes,
    tree_method="hist", n_jobs=-1, random_state=SEED,
    eval_metric="mlogloss",
)
xgb.fit(X_tr, y_train, sample_weight=sample_w)
xgb_train_time = time.perf_counter() - t0
rep_xgb, f1_xgb, gmean_xgb = eval_sklearn(xgb, X_te, y_test, class_names)
print(f"Waktu latih: {xgb_train_time:.2f}s | macro-F1={f1_xgb:.4f} | G-mean={gmean_xgb:.4f}")
display(rep_xgb)
ensemble_results["XGBoost"] = {
    "macro_f1": f1_xgb, "g_mean": gmean_xgb, "train_time": xgb_train_time,
    "size_mb": sklearn_model_size_mb(xgb), "latency_ms": measure_sklearn_latency(xgb, X_te),
    "report": rep_xgb,
}

In [ ]:
# ---------- Soft-voting ensemble (RF + XGBoost) ----------
print("=== Melatih Voting Ensemble (RF + XGBoost) ===")
t0 = time.perf_counter()
voting = VotingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=100, n_jobs=-1,
                                      class_weight="balanced", random_state=SEED)),
        ("xgb", XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.3,
                              objective="multi:softprob", num_class=n_classes,
                              tree_method="hist", n_jobs=-1, random_state=SEED,
                              eval_metric="mlogloss")),
    ],
    voting="soft", n_jobs=1,
)
voting.fit(X_tr, y_train)
vote_train_time = time.perf_counter() - t0
rep_vote, f1_vote, gmean_vote = eval_sklearn(voting, X_te, y_test, class_names)
print(f"Waktu latih: {vote_train_time:.2f}s | macro-F1={f1_vote:.4f} | G-mean={gmean_vote:.4f}")
display(rep_vote)
ensemble_results["Voting (RF+XGB)"] = {
    "macro_f1": f1_vote, "g_mean": gmean_vote, "train_time": vote_train_time,
    "size_mb": sklearn_model_size_mb(voting), "latency_ms": measure_sklearn_latency(voting, X_te),
    "report": rep_vote,
}

### 13.1 Tabel Perbandingan Menyeluruh: Performa vs Biaya Komputasi

Tabel inilah yang menjadi argumen utama untuk paper. Yang dinilai bukan sekadar siapa yang macro-F1-nya
tertinggi, melainkan **rasio antara keuntungan performa dan biaya yang harus dibayar**.

Perlu dicatat secara jujur di paper: tree ensemble dilatih pada CPU multi-core sedangkan Transformer
dilatih pada GPU, sehingga waktu latih keduanya **tidak sepenuhnya sebanding**. Yang lebih adil
dibandingkan adalah ukuran model dan latensi inferensi, karena keduanya diukur pada kondisi CPU.

In [ ]:
# =====================================================================
# TABEL 3 (multi-seed): performa + biaya komputasi, rerata ± simpangan baku
# =====================================================================
# Versi sebelumnya menghitung tabel ini dari SATU seed, sementara Tabel 1 dan 2
# memakai banyak seed. Ketidakkonsistenan itu terlihat jelas oleh reviewer, dan
# pada seed tertentu F1 kelas U2R bisa menyimpang jauh dari reratanya.
# Di sini tree ensemble dan SMOTE diulang pada SEED YANG SAMA dengan Tabel 1.

from imblearn.over_sampling import SMOTE as _SMOTE

TAB3_SEEDS = ABL_SEEDS
print(f"Menjalankan Tabel 3 pada {len(TAB3_SEEDS)} seed: {TAB3_SEEDS}\n")

def _minf1(rep, cls):
    row = rep.loc[rep["class"] == cls, "f1"]
    return float(row.values[0]) if len(row) else np.nan

tab3 = {}          # nama model -> list of (macroF1, gmean, f1_min0, f1_min1)
tab3_cost = {}     # nama model -> dict biaya (diukur sekali, deterministik)
smote_overhead = []

for seed in TAB3_SEEDS:
    print(f"--- Seed {seed} ---")

    # ---------- Tree ensemble ----------
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1,
                                class_weight="balanced", random_state=seed)
    rf.fit(X_tr, y_train)

    cls_counts = np.bincount(y_train, minlength=n_classes).astype(float)
    sample_w = (len(y_train) / (n_classes * np.maximum(cls_counts, 1)))[y_train]
    xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.3,
                        objective="multi:softprob", num_class=n_classes,
                        tree_method="hist", n_jobs=-1, random_state=seed,
                        eval_metric="mlogloss")
    xgb.fit(X_tr, y_train, sample_weight=sample_w)

    voting = VotingClassifier(
        estimators=[("rf", RandomForestClassifier(n_estimators=100, n_jobs=-1,
                                                  class_weight="balanced", random_state=seed)),
                    ("xgb", XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.3,
                                          objective="multi:softprob", num_class=n_classes,
                                          tree_method="hist", n_jobs=-1, random_state=seed,
                                          eval_metric="mlogloss"))],
        voting="soft", n_jobs=1)
    voting.fit(X_tr, y_train)

    for nm, mdl in [("Random Forest", rf), ("XGBoost", xgb), ("Voting (RF+XGB)", voting)]:
        rep, mf1, gm = eval_sklearn(mdl, X_te, y_test, class_names)
        tab3.setdefault(nm, []).append((mf1, gm,
                                        _minf1(rep, minority_names[0]),
                                        _minf1(rep, minority_names[1])))
        if nm not in tab3_cost:
            tab3_cost[nm] = {"size": sklearn_model_size_mb(mdl),
                             "lat": measure_sklearn_latency(mdl, X_te)}
        print(f"  {nm:18s} macroF1={mf1:.4f}")

    # ---------- SMOTE ----------
    t0 = time.perf_counter()
    mc = pd.Series(y_train).value_counts().min()
    sm = _SMOTE(random_state=seed, k_neighbors=min(5, max(1, mc - 1)))
    Xr, yr = sm.fit_resample(X_train_combined, y_train)
    smote_overhead.append((time.perf_counter() - t0, len(yr)))

    if n_categorical > 0:
        Xr_num = Xr[:, :X_train_num.shape[1]]
        Xr_cat = np.rint(Xr[:, X_train_num.shape[1]:]).astype(np.int64)
        for j, card in enumerate(cat_cardinalities):
            Xr_cat[:, j] = np.clip(Xr_cat[:, j], 0, card - 1)
    else:
        Xr_num, Xr_cat = Xr, np.zeros((len(yr), 0), dtype=np.int64)

    sm_loader = torch.utils.data.DataLoader(
        IDSDataset(Xr_num, Xr_cat, yr), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    sm_model = BaselineIDSTransformer(n_numeric, cat_cardinalities, n_classes,
                                      d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS)
    sm_model = train_model(sm_model, sm_loader, test_loader, nn.CrossEntropyLoss(), verbose=False)
    rep, mf1, gm, _, _ = evaluate(sm_model, test_loader, class_names)
    tab3.setdefault("SMOTE", []).append((mf1, gm,
                                         _minf1(rep, minority_names[0]),
                                         _minf1(rep, minority_names[1])))
    if "SMOTE" not in tab3_cost:
        tab3_cost["SMOTE"] = {"size": model_size_mb(sm_model),
                              "lat": measure_latency(sm_model, test_loader, single_thread=True)[0]}
        sm_model.to(device)
    print(f"  {'SMOTE':18s} macroF1={mf1:.4f}")

    # Simpan parsial setiap seed
    pd.DataFrame([{"seed": sd, "model": nm, "macro_f1": r[0], "g_mean": r[1],
                   f"f1_{minority_names[0]}": r[2], f"f1_{minority_names[1]}": r[3]}
                  for nm, rs in tab3.items()
                  for sd, r in zip(TAB3_SEEDS, rs)]).to_csv(f"tabel3_partial_{TAG}.csv", index=False)

# ---------- Varian Transformer diambil dari ablasi (sudah multi-seed) ----------
DL_COST = {
    "ClassAttn + Focal": proposed_model,
    "Focal":             baseline_model,   # arsitektur sama dengan Focal-only
    "ClassAttn":         proposed_model,
    "Baseline":          baseline_model,
}
NAME_MAP = {"Full (Attn+Focal)": "ClassAttn + Focal", "Focal-only": "Focal",
            "Attn-only": "ClassAttn", "Baseline": "Baseline"}
for abl_name, disp in NAME_MAP.items():
    tab3[disp] = [tuple(r) for r in abl_arr[abl_name]]
    m = DL_COST[disp]
    tab3_cost[disp] = {"size": model_size_mb(m),
                       "lat": measure_latency(m, test_loader, single_thread=True)[0]}
    m.to(device)

# ---------- Rakit tabel ----------
rows = []
for nm, vals in tab3.items():
    a = np.array(vals)
    rows.append({
        "Model": nm,
        "Macro-F1": f"{a[:,0].mean():.4f} ± {a[:,0].std():.4f}",
        "G-mean": f"{a[:,1].mean():.4f} ± {a[:,1].std():.4f}",
        f"F1 ({minority_names[0]})": f"{a[:,2].mean():.4f} ± {a[:,2].std():.4f}",
        f"F1 ({minority_names[1]})": f"{a[:,3].mean():.4f} ± {a[:,3].std():.4f}",
        "Size (MB)": round(tab3_cost[nm]["size"], 4),
        "Latency (ms)": round(tab3_cost[nm]["lat"], 2),
        "_sort": a[:,0].mean(),
    })

full_comparison = (pd.DataFrame(rows).sort_values("_sort", ascending=False)
                   .drop(columns="_sort").reset_index(drop=True))
display(full_comparison)
full_comparison.to_csv(f"tabel3_multiseed_{TAG}.csv", index=False)

ov = np.array(smote_overhead)
print(f"\nOverhead SMOTE: {ov[:,0].mean():.1f} detik/seed; "
      f"baris data latih {len(y_train):,} -> {int(ov[:,1].mean()):,} "
      f"({ov[:,1].mean()/len(y_train):.2f}x)")
print("Biaya (Size, Latency) deterministik, diukur sekali per model.")
print(f"Tersimpan: tabel3_multiseed_{TAG}.csv")

In [ ]:
# Visualisasi trade-off: performa kelas minoritas vs ukuran model
import matplotlib.pyplot as plt

mcol = f"F1 ({minority_names[0]})"
plot_df = full_comparison.copy()
# kolom sekarang "mean ± std" -> ambil reratanya saja untuk diplot
plot_df["_x"] = plot_df["Size (MB)"].astype(float)
plot_df["_y"] = plot_df[mcol].astype(str).str.split("±").str[0].astype(float)

TREE = {"Random Forest", "XGBoost", "Voting (RF+XGB)"}
fig, ax = plt.subplots(figsize=(9, 6))
for _, r in plot_df.iterrows():
    tipe = "Tree ensemble" if r["Model"] in TREE else "Transformer"
    ax.scatter(r["_x"], r["_y"], s=160,
               color="#E4572E" if tipe == "Tree ensemble" else "#2E86AB",
               edgecolors="black", zorder=3)
    ax.annotate(r["Model"], (r["_x"], r["_y"]), textcoords="offset points",
                xytext=(8, 6), fontsize=8)

ax.set_xscale("log")
ax.set_xlabel("Ukuran model (MB, skala log)")
ax.set_ylabel(f"F1-score kelas minoritas ({minority_names[0]}), rerata {len(ABL_SEEDS)} seed")
ax.set_title("Trade-off: performa kelas minoritas vs ukuran model")
ax.grid(alpha=0.3, zorder=0)
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c,
                      markeredgecolor="black", markersize=10, label=k)
           for k, c in {"Transformer": "#2E86AB", "Tree ensemble": "#E4572E"}.items()]
ax.legend(handles=handles)
plt.tight_layout()
plt.savefig(f"fig3_tradeoff_{TAG}.png", dpi=300, bbox_inches="tight", facecolor="white")
print(f"Tersimpan: fig3_tradeoff_{TAG}.png")
plt.show()

print("Titik di kiri-atas paling diinginkan: performa tinggi dengan model kecil.")

## Catatan Penutup

- Notebook ini adalah **implementasi kerangka kerja** sesuai bagian Metodologi draft jurnal. Untuk hasil publikasi: tuning hyperparameter (`d_model`, `n_layers`, `gamma`, `beta`), tambah epoch, dan pertimbangkan k-fold cross-validation.
- **Framing klaim.** Efisiensi di notebook ini diukur melalui parameter, FLOPs, ukuran model, latensi single-thread, dan quantization — semuanya *computational efficiency* pada simulasi CPU terbatas. Jangan menulis "edge deployment" atau "IoT device" di paper kecuali kamu benar-benar menguji pada perangkat fisik.
- **Signifikansi statistik.** Notebook ini memakai `N_SEEDS = 10`. Dengan `N_SEEDS = 5`, p-value terkecil yang mungkin dicapai Wilcoxon adalah 0.0625 — artinya klaim "p < 0.05" mustahil. Naikkan ke minimal 6 seed, idealnya 10.
- **Catatan SMOTE.** Implementasi di sini menerapkan SMOTE biasa pada fitur campuran (kategorikal diperlakukan sebagai numerik terenkode lalu dibulatkan). Untuk paper, sebutkan keterbatasan ini atau ganti dengan SMOTE-NC.
- **Catatan baseline ensemble.** Tree ensemble dilatih di CPU multi-core sementara Transformer di GPU, sehingga waktu latih keduanya tidak sepenuhnya sebanding — bandingkan ukuran model dan latensi inferensi saja. Bersiaplah pada kemungkinan Random Forest atau XGBoost mengungguli Transformer pada macro-F1; pada data tabular hal ini lazim terjadi. Bila itu yang terjadi, laporkan apa adanya dan geser argumen paper ke performa kelas minoritas, bukan performa rata-rata.
- Untuk generalisasi lintas dataset, jalankan seluruh notebook dua kali dengan `DATASET_CHOICE = "NSLKDD"` lalu `"CICIDS2017"`, kemudian bandingkan tabel hasilnya.
- Modul **knowledge distillation** belum disertakan agar notebook tetap ringkas; dapat ditambahkan dengan melatih versi besar `BaselineIDSTransformer` sebagai teacher lalu menambahkan loss KL-divergence terhadap output student.
- **Tidak ada seleksi model pada test set.** `train_model()` melatih sejumlah epoch tetap
  dan mengevaluasi test set satu kali di akhir. Versi sebelumnya memilih bobot dengan
  macro-F1 test tertinggi per epoch — itu test-set leakage dan membuat seluruh angka
  bias optimistis. Jangan kembalikan perilaku tersebut.